# Normalização e Agrupamento de Termos Clínicos (Diplotokens)

Este notebook implementa e valida a **Função Final** (`reconstruct_clinical_tokens`) que resolve os dois maiores gargalos de tokenização identificados no projeto:
1. **Expressões Multipalavras (Doenças, Sintomas, Procedimentos):** Tokens individuais contíguos que foram separados por espaços no regex são reunidos em uma única entidade semântica (ex: `['lower', 'quadrant', 'abdominal', 'pain']` → `'lower quadrant abdominal pain'`).
2. **Siglas e Acrônimos Clínicos:** Mapeamento de termos curtos em caixa alta para suas formas completas canônicas (ex: `GDC` → `GDC [gastric duplication cyst]`).
3. **Preservação de Escalares:** Mantém números, pontuação e medições compostas intactas (ex: `12,476.5ng/ml`, `3-day`, `44`).

---

### A Lematização entra antes ou depois do agrupamento?
* **Lematização dos tokens ANTES da busca de termos:** Plurais flexionados (`cysts`, `lesions`, `masses`) são convertidos para o singular antes de checar contra o vocabulário, pois os vocabulários médicos (MeSH, CID-10, SNOMED) registram os termos na forma canônica no singular (`cystic lesion`).
* **Proteção de Siglas:** Tokens em caixa alta (`GDC`, `CEA`, `SLE`) **não** sofrem minúsculas nem lematização agressiva, evitando perda de informação.


In [1]:
import re
import json
from pathlib import Path
import pandas as pd

# Carrega os dados de amostra
cases_path = Path("../data/raw/sample/cases.csv")
meta_path = Path("../data/raw/sample/metadata.csv")

cases_df = pd.read_csv(cases_path)
meta_df = pd.read_csv(meta_path)

print(f"Total de casos carregados: {len(cases_df)}")
print(f"Total de artigos em metadata: {len(meta_df)}")
cases_df.head(2)[["case_id", "age", "gender"]]


Total de casos carregados: 56
Total de artigos em metadata: 50


In [2]:
# ==============================================================================
# TOKENIZADOR CLÍNICO POR EXPRESSÕES REGULARES
# (Mesma implementação consolidada em exploracao_polas.ipynb e saída do sla.txt)
# ==============================================================================

UNIT = r'(?:kg|mg|mcg|ng|iu|U|L|mmHg|mmol|ml|dl|l|cm|mm|g|hg|min|y/o|yo|yrs?|years?|months?|weeks?|days?)'
TIME_UNIT = r'(?:min|hr|hrs|h|s|sec|m)'

TOKEN_SEP_REGEX = re.compile(rf'''
    \d+(?:,\d{{3}})*(?:\.\d+)?                                                # números com vírgula e ponto decimal
    (?:
        /\d+(?:,\d{{3}})*(?:\.\d+)?(?:\s?{UNIT}(?:\s?{UNIT})?)?(?![A-Za-z])   # razão: núm/núm + unidade opcional
      | /{TIME_UNIT}(?![A-Za-z])                                              # taxa: núm/unidade-de-tempo
      | \s?{UNIT}(?:/{UNIT})?(?![A-Za-z])                                     # número + unidade simples/composta
    )?
  | [A-Za-z]+(?:-[A-Za-z]+)*                                                  # palavras simples e hifenizadas
  | [.,;:()]                                                                   # pontuação como token isolado
''', re.VERBOSE | re.IGNORECASE)

def find_tokens(text: str) -> list[str]:
    """Segmenta o texto bruto em lista de tokens básicos."""
    return [match.group(0) for match in TOKEN_SEP_REGEX.finditer(text)]

print("Tokenizador carregado com sucesso!")


Tokenizador carregado com sucesso!


In [3]:
# ==============================================================================
# VOCABULÁRIO MÉDICO E TABELA DE ACRÔNIMOS
# ==============================================================================

# 1. Siglas clínicas frequentes extraídas das definições dos próprios casos
ACRONYM_MAP = {
    "GDC": "gastric duplication cyst",
    "CEA": "carcinoembryonic antigen",
    "CA": "carbohydrate antigen",
    "EUS-FNA": "endoscopic ultrasound-guided fine needle aspiration",
    "FNA": "fine needle aspiration",
    "CT": "computed tomography",
    "MRI": "magnetic resonance imaging",
    "VSD": "ventricular septal defect",
    "SLE": "systemic lupus erythematosus",
    "TTP": "thrombotic thrombocytopenic purpura",
    "HTN": "hypertension",
    "DM2": "diabetes mellitus type 2",
    "IM": "internal medicine",
    "ER": "emergency room",
    "UC": "ulcerative colitis",
}

# 2. Construção do vocabulário médico (MeSH + Keywords + Termos Clínicos Centrais)
def build_medical_lexicon(metadata_df: pd.DataFrame) -> set[str]:
    lexicon = set()
    
    # Termos clínicos centrais (sintomas, anatomia, procedimentos e patologia)
    core_terms = {
        "lower quadrant abdominal pain", "right flank and lower quadrant abdominal pain",
        "right flank", "abdominal pain", "nausea", "constipation",
        "contrast enhanced computed tomography", "computed tomography",
        "cystic lesion", "pancreas", "stomach", "pancreatic echotexture",
        "internal septations", "associated masses", "extracellular mucin",
        "carcinoembryonic antigen", "carbohydrate antigen",
        "mucinous pancreatic cystic neoplasm", "mucinous pancreatic cyst",
        "laparoscopic distal pancreatectomy", "surgical resection",
        "lesser sac", "posterior wall", "intraoperative endoscopy",
        "coeliac axis", "coeliac vessels", "gastric duplication cyst",
        "surgical stapler", "oesophagogastroduodenoscopy",
        "ventricular septal defect", "infective endocarditis",
        "systemic lupus erythematosus", "chest pain", "fever",
        "shortness of breath", "dyspnea", "headache", "malignancy"
    }
    lexicon.update(core_terms)
    
    # Extrai termos do metadata.csv (MeSH e Keywords de todos os 50 artigos)
    for _, row in metadata_df.iterrows():
        # Keywords
        kw = row.get("keywords")
        if pd.notna(kw) and str(kw).startswith("["):
            for item in kw[1:-1].split(","):
                clean = item.strip().strip("'").strip('"').lower()
                if len(clean) > 2:
                    lexicon.add(clean)
        # MeSH terms
        mesh = row.get("mesh_terms")
        if pd.notna(mesh) and str(mesh).startswith("["):
            for item in mesh[1:-1].split(","):
                clean = item.split("/")[0].strip().strip("'").strip('"').lower()
                if len(clean) > 2 and clean not in {"case reports", "humans", "male", "female", "adult", "middle aged", "aged"}:
                    lexicon.add(clean)
                    
    return lexicon

MEDICAL_LEXICON = build_medical_lexicon(meta_df)
print(f"Total de conceitos médicos no léxico: {len(MEDICAL_LEXICON)}")
print("Exemplos de termos multipalavras cadastrados:")
sample_compounds = sorted(list(filter(lambda x: ' ' in x, list(MEDICAL_LEXICON))))[:10]
for term in sample_compounds:
    print(f"  - {term}")


Total de conceitos médicos no léxico: 236
Exemplos de termos multipalavras cadastrados:
  - 17α-hydroxylase deficiency
  - abdominal pain
  - acute appendicitis
  - adenovirus infections
  - alasdair macintyre
  - anatomical variations
  - anticoagulation therapy
  - aortic replacement
  - appendiceal endometriosis
  - associated masses


In [4]:
# ==============================================================================
# FUNÇÃO FINAL: LEMATIZAÇÃO SELETIVA + CASAMENTO GULOSO DE N-GRAMAS
# ==============================================================================

IRREGULAR_LEMMAS = {
    "cysts": "cyst", "lesions": "lesion", "masses": "mass",
    "examinations": "examination", "rates": "rate", "antibodies": "antibody",
    "defects": "defect", "infections": "infection", "seizures": "seizure",
    "stenoses": "stenosis", "thrombi": "thrombus", "diverticula": "diverticulum"
}

def lemmatize_token(token: str) -> str:
    """Lematiza substantivos e termos médicos simples preservando siglas em caixa alta."""
    if token.isupper() and len(token) >= 2:
        return token
    lower = token.lower()
    if lower in IRREGULAR_LEMMAS:
        return IRREGULAR_LEMMAS[lower]
    if lower.endswith("ies") and len(lower) > 4:
        return lower[:-3] + "y"
    if lower.endswith("es") and len(lower) > 3 and lower[-3] in "shxz":
        return lower[:-2]
    if lower.endswith("s") and not lower.endswith("ss") and len(lower) > 3:
        return lower[:-1]
    return lower

def reconstruct_clinical_tokens(
    tokens: list[str],
    vocabulary: set[str],
    acronym_map: dict[str, str] = ACRONYM_MAP,
    expand_acronyms: bool = True,
    max_n: int = 5
) -> list[str]:
    """
    FUNÇÃO FINAL DO PIPELINE:
    Recebe a lista de tokens do tokenizador e produz a lista com correções:
    - Agrupa sequências multipalavras que representam entidades médicas (longest-match).
    - Lematiza plurais para casar com a ontologia/dicionário.
    - Resolve e expande siglas médicas.
    - Mantém números, unidades escalares e pontuação intactos.
    """
    n = len(tokens)
    result = []
    i = 0

    while i < n:
        tok = tokens[i]

        # 1. Tratamento de Siglas/Acrônimos
        if tok in acronym_map:
            expanded = f"{tok} [{acronym_map[tok]}]" if expand_acronyms else acronym_map[tok]
            result.append(expanded)
            i += 1
            continue

        # 2. Janela Deslizante Gulosa para Expressões Multipalavras (de max_n até 2 tokens)
        matched = False
        for k in range(min(max_n, n - i), 1, -1):
            window = tokens[i : i + k]
            
            # Não permite cruzar pontuações fortes ou parênteses
            if any(p in window for p in [".", ";", "!", "?", "(", ")"]):
                continue

            # Testa correspondência literal e lematizada
            raw_phrase = " ".join(window).lower()
            lemma_phrase = " ".join(lemmatize_token(t) for t in window)

            if raw_phrase in vocabulary:
                result.append(" ".join(window))
                i += k
                matched = True
                break
            elif lemma_phrase in vocabulary:
                result.append(" ".join(window))
                i += k
                matched = True
                break

        if matched:
            continue

        # 3. Termo simples não-composto
        result.append(tok)
        i += 1

    return result

print("Função 'reconstruct_clinical_tokens' compilada e pronta para uso!")


Função 'reconstruct_clinical_tokens' compilada e pronta para uso!


In [5]:
# ==============================================================================
# TESTE 1 (HARDCODED): SENTENÇA INICIAL DO CASO 1 COM ASSERÇÕES
# ==============================================================================

sentence_1 = (
    "A 44-year-old woman presented with a 3-day history of right flank "
    "and lower quadrant abdominal pain associated with nausea and constipation."
)

print("=" * 80)
print("SENTENÇA DE ENTRADA (CASO 1):")
print(sentence_1)
print("=" * 80)

# 1. Tokenização inicial (como saía em sla.txt)
tokens_1 = find_tokens(sentence_1)
print(f"\n[1] TOKENS SOLTOS (Tokenizador Inicial - {len(tokens_1)} tokens):")
print(" | ".join(tokens_1))

# 2. Aplicação da função final
reconstructed_1 = reconstruct_clinical_tokens(tokens_1, MEDICAL_LEXICON)
print(f"\n[2] TOKENS RECONSTRUÍDOS ({len(reconstructed_1)} tokens):")
print(" | ".join(reconstructed_1))

# 3. Termos agrupados em destaque
print("\n[3] ENTIDADES COMPOSTAS AGRUPADAS:")
for t in reconstructed_1:
    if " " in t:
        print(f"  -> '{t}'")

# ==========================================
# ASSERÇÕES DE VALIDAÇÃO (ASSERT)
# ==========================================
assert "lower quadrant abdominal pain" in reconstructed_1, "Erro: 'lower quadrant abdominal pain' deveria estar agrupado!"
assert "right flank" in reconstructed_1, "Erro: 'right flank' deveria estar agrupado!"
assert "nausea" in reconstructed_1, "Erro: 'nausea' deveria estar presente!"
assert "constipation" in reconstructed_1, "Erro: 'constipation' deveria estar presente!"
assert "44" in reconstructed_1 and "year-old" in reconstructed_1, "Erro: demografia deve ser preservada!"

print("\n" + "=" * 80)
print("✓ TODAS AS ASSERÇÕES DO TESTE 1 PASSARAM COM SUCESSO!")
print("=" * 80)


SENTENÇA DE ENTRADA (CASO 1):
A 44-year-old woman presented with a 3-day history of right flank and lower quadrant abdominal pain associated with nausea and constipation.

[1] TOKENS SOLTOS (Tokenizador Inicial - 24 tokens):
A | 44 | year-old | woman | presented | with | a | 3 | day | history | of | right | flank | and | lower | quadrant | abdominal | pain | associated | with | nausea | and | constipation | .

[2] TOKENS RECONSTRUÍDOS (20 tokens):
A | 44 | year-old | woman | presented | with | a | 3 | day | history | of | right flank | and | lower quadrant abdominal pain | associated | with | nausea | and | constipation | .

[3] ENTIDADES COMPOSTAS AGRUPADAS:
  -> 'right flank'
  -> 'lower quadrant abdominal pain'

✓ TODAS AS ASSERÇÕES DO TESTE 1 PASSARAM COM SUCESSO!


In [6]:
# ==============================================================================
# TESTE 2 (HARDCODED): EXAMES, MEDIDAS DE LABORATÓRIO E DIAGNÓSTICO
# ==============================================================================

sentence_2 = (
    "FNA of the cyst demonstrated no evidence of malignancy but did show the presence of "
    "extracellular mucin as well as a carcinoembryonic antigen (CEA) level of 12,476.5ng/ml "
    "and a carbohydrate antigen (CA) 19-9 level of 6iu/ml, suggesting the diagnosis of a "
    "mucinous pancreatic cystic neoplasm."
)

print("=" * 80)
print("SENTENÇA DE ENTRADA (EXAMES E DIAGNÓSTICO):")
print(sentence_2)
print("=" * 80)

tokens_2 = find_tokens(sentence_2)
reconstructed_2 = reconstruct_clinical_tokens(tokens_2, MEDICAL_LEXICON)

print(f"\nTOKENS RECONSTRUÍDOS ({len(reconstructed_2)} tokens):")
print(" | ".join(reconstructed_2))

# Asserções
assert "mucinous pancreatic cystic neoplasm" in reconstructed_2
assert "extracellular mucin" in reconstructed_2
assert "carcinoembryonic antigen" in reconstructed_2
assert "12,476.5ng/ml" in reconstructed_2
assert "6iu/ml" in reconstructed_2

# Sentença com Sigla GDC e Patologia Final
sentence_3 = (
    "Final pathology revealed a 9.5cm x 4.5cm x 2.0cm cyst with a smooth internal cyst wall "
    "consistent with a GDC (Figs 3-5)."
)
print("\n" * 1 + "=" * 80)
print("TESTE 3: Resolução de Sigla GDC e Patologia")
tokens_3 = find_tokens(sentence_3)
reconstructed_3 = reconstruct_clinical_tokens(tokens_3, MEDICAL_LEXICON)
print(" | ".join(reconstructed_3))

assert any("GDC" in t for t in reconstructed_3), "Erro: Sigla GDC deveria ter sido tratada!"
print("\n" + "=" * 80)
print("✓ TODAS AS ASSERÇÕES DE EXAMES E DIAGNÓSTICO PASSARAM COM SUCESSO!")
print("=" * 80)


SENTENÇA DE ENTRADA (EXAMES E DIAGNÓSTICO):
FNA of the cyst demonstrated no evidence of malignancy but did show the presence of extracellular mucin as well as a carcinoembryonic antigen (CEA) level of 12,476.5ng/ml and a carbohydrate antigen (CA) 19-9 level of 6iu/ml, suggesting the diagnosis of a mucinous pancreatic cystic neoplasm.

TOKENS RECONSTRUÍDOS (46 tokens):
FNA [fine needle aspiration] | of | the | cyst | demonstrated | no | evidence | of | malignancy | but | did | show | the | presence | of | extracellular mucin | as | well | as | a | carcinoembryonic antigen | ( | CEA [carcinoembryonic antigen] | ) | level | of | 12,476.5ng/ml | and | a | carbohydrate antigen | ( | CA [carbohydrate antigen] | ) | 19 | 9 | level | of | 6iu/ml | , | suggesting | the | diagnosis | of | a | mucinous pancreatic cystic neoplasm | .

TESTE 3: Resolução de Sigla GDC e Patologia
Final | pathology | revealed | a | 9.5cm | x | 4.5cm | x | 2.0cm | cyst | with | a | smooth | internal | cyst | wall | co

In [7]:
# ==============================================================================
# CÉLULA INTERATIVA: TESTE COM INPUT PERSONALIZADO
# ==============================================================================

default_clinical_phrase = (
    "The patient had a past medical history of systemic lupus erythematosus and "
    "underwent contrast enhanced computed tomography for ventricular septal defect."
)

# Para digitar interativamente no notebook, descomente a linha abaixo:
# user_input = input("Digite uma sentença clínica (ou tecle Enter para usar a padrão): ").strip() or default_clinical_phrase
user_input = default_clinical_phrase

print("Sentença em Análise:")
print(user_input)
print("-" * 80)

custom_raw_tokens = find_tokens(user_input)
custom_reconstructed = reconstruct_clinical_tokens(custom_raw_tokens, MEDICAL_LEXICON)

print(f"\nTokens Antes ({len(custom_raw_tokens)}):")
print(" | ".join(custom_raw_tokens))

print(f"\nTokens Depois ({len(custom_reconstructed)}):")
print(" | ".join(custom_reconstructed))

print("\nTermos Clínicos Identificados e Unificados:")
for t in custom_reconstructed:
    if " " in t or "[" in t:
        print(f"  ★ {t}")


Sentença em Análise:
The patient had a past medical history of systemic lupus erythematosus and underwent contrast enhanced computed tomography for ventricular septal defect.
--------------------------------------------------------------------------------

Tokens Antes (22):
The | patient | had | a | past | medical | history | of | systemic | lupus | erythematosus | and | underwent | contrast | enhanced | computed | tomography | for | ventricular | septal | defect | .

Tokens Depois (15):
The | patient | had | a | past | medical | history | of | systemic lupus erythematosus | and | underwent | contrast enhanced computed tomography | for | ventricular septal defect | .

Termos Clínicos Identificados e Unificados:
  ★ systemic lupus erythematosus
  ★ contrast enhanced computed tomography
  ★ ventricular septal defect


In [8]:
# ==============================================================================
# EXECUÇÃO NO CASO 1 COMPLETO (Texto Integral)
# ==============================================================================

case1_full_text = cases_df.loc[0, "case_text"]
raw_c1_tokens = find_tokens(case1_full_text)
proc_c1_tokens = reconstruct_clinical_tokens(raw_c1_tokens, MEDICAL_LEXICON)

print(f"Resultados no Caso 1 ({cases_df.loc[0, 'case_id']}) Completo:")
print(f"- Tokens brutos no sla.txt: {len(raw_c1_tokens)}")
print(f"- Tokens finais unificados: {len(proc_c1_tokens)}")
print(f"- Redução de fragmentação: {len(raw_c1_tokens) - len(proc_c1_tokens)} tokens agrupados em conceitos clínicos")

print("\nCatálogo de Entidades Clínicas Reconstruídas no Caso 1:")
uniques = sorted(set([t for t in proc_c1_tokens if " " in t or "[" in t]))
for term in uniques:
    print(f"  • {term}")


Resultados no Caso 1 (PMC5137649_01) Completo:
- Tokens brutos no sla.txt: 420
- Tokens finais unificados: 390
- Redução de fragmentação: 30 tokens agrupados em conceitos clínicos

Catálogo de Entidades Clínicas Reconstruídas no Caso 1:
  • CA [carbohydrate antigen]
  • CEA [carcinoembryonic antigen]
  • EUS-FNA [endoscopic ultrasound-guided fine needle aspiration]
  • FNA [fine needle aspiration]
  • GDC [gastric duplication cyst]
  • Intraoperative endoscopy
  • associated masses
  • carbohydrate antigen
  • carcinoembryonic antigen
  • coeliac axis
  • coeliac vessels
  • contrast enhanced computed tomography
  • cystic lesion
  • extracellular mucin
  • gastric duplication cyst
  • internal septations
  • laparoscopic distal pancreatectomy
  • lesser sac
  • lower quadrant abdominal pain
  • mucinous pancreatic cystic neoplasm
  • pancreatic echotexture
  • posterior wall
  • right flank
  • surgical resection
  • surgical stapler
